In [ ]:
from pathlib import Path
import sys

import numpy as np
import matplotlib.pyplot as plt

_repo = Path.cwd()
while _repo != _repo.parent and not (_repo / "pyproject.toml").exists():
    _repo = _repo.parent
sys.path.insert(0, str(_repo))

from syssimx_examples.controlled_pendulum.components import (
    FMUPendulum, OpenSimPendulum, FEMPendulum
)
from demos.ControlledPendulum.scripts.torque_profiles import (
    constant_torque, ramp_torque, zero_torque
)
from syssimx_examples.controlled_pendulum import MasterPendulum
import syssimx_examples.controlled_pendulum.components.fem.pendulum_config as config
from thesis.notebooks.plot_setup import set_professional_style
plt = set_professional_style()

from syssimx import Connection, System, FMUComponent
from syssimx.system.connection import EventConnection
from syssimx.viz.system_graph_visualizer import SystemGraphVisualizer

In [ ]:
mesh_params = config.MeshParameters()

init_params = config.InitialConditionParameters()
init_params.angular_position_deg = np.rad2deg(0.3) # Initial angle
q0 = np.deg2rad(init_params.angular_position_deg)
init_params.drive_torque = 0 # Nm

mat_params = config.MaterialParameters()
mat_params.E_pendulum = 2.1e11  # Young's modulus for the pendulum
mat_params.nu_pendulum = 0.3    # Poisson's ratio for the pendulum
mat_params.rho_pendulum = 2700  # Density for the pendulum

sim_params = config.SimulationParameters()
sim_params.tau = 0.01
sim_params.t_end = 1.0
sim_params.with_contact = False
sim_params.use_gravity = True

contact_params = config.ContactParameters()
contact_params.kn = 1e10

anim_params = config.AnimationParameters()
anim_params.animate = True

fem_parameters = {
    'mat_params': mat_params,
    'contact_params': contact_params,
    'init_params': init_params,
    'sim_params': sim_params,
    'anim_params': anim_params,
    'mesh_params': mesh_params,
}

pendulum = MasterPendulum(name="MasterPendulum", initial_mode="FMU")

pendulum.set_parameters(**{"FEM": fem_parameters})
pendulum.initialize(t0=0.0)

t = 0.0
dt = pendulum.fem.sim_params.tau
t_end = pendulum.fem.sim_params.t_end

In [ ]:
from OMPython import ModelicaSystem

# 1) Build the Modelica Model
pkg_dir_path = _repo / "demos/ControlledPendulum/src/modelica/ControlledPendulum/"
pkg_file_path = pkg_dir_path / "package.mo"
model = "ControlledPendulum.Plants.PendulumWithDiscreteWall"

# 2) Create Modelica System Model
model = ModelicaSystem(str(pkg_file_path), model)

In [ ]:
model.getParameters()

In [ ]:
# 3) Set Model Parameters
parameters = {  
    f"L":f"{pendulum.fem._equivalent_length}",
    f"m":f"{pendulum.fem.mass}",
    f"J":f"{pendulum.fem.inertia}",
    f"theta_start":f"{q0}",
}
model.setParameters(parameters)

# 4) Set simulation options
t0 = 0.0
dt_monolithic = 1e-4
solver_tolerance = 1e-8
solver_monolithic = "dassl"
simulation_options = {
    "startTime": f"{t0}",
    "stopTime": f"{t_end}",
    "stepSize": f"{dt_monolithic}",
    "tolerance": f"{solver_tolerance}",
    "solver": f"{solver_monolithic}",
}
model.setSimulationOptions(simulation_options)

# 5) Build model
model.buildModel()

# 6) Run simulation
model.simulate()

# 7) Retrieve results
ref_monolithic = {}
keys = ("time", "theta", "omega", "alpha")
for key in keys:
    print("Extracting: ", key)
    ref_monolithic[key] = model.getSolutions(key)
    ref_monolithic[key] = ref_monolithic[key].flatten()

t_ref = ref_monolithic["time"]
q_ref = ref_monolithic["theta"]
omega_ref = ref_monolithic["omega"]
alpha_ref = ref_monolithic["alpha"]

In [ ]:
def wall_contact_indicator(comp) -> float:
    theta = comp.get_outputs()["theta"]
    threshold = np.deg2rad(0)  # 0 degrees in radians
    return theta - threshold

pendulum.add_event_indicator("wall_hit", wall_contact_indicator, -1)

system = System(name="Master Pendulum System")
system.add_component(pendulum)

event_connection = EventConnection(
    src_comp=pendulum.name,
    src_port=pendulum.output_specs["wall_hit"].name,
    dst_comp=pendulum.name,
    dst_port=pendulum.input_specs["omega_invert"].name,
)

system.add_event_connection(event_connection)
system.initialize(0.0)
pendulum.switch_regions

In [ ]:
visualizer = SystemGraphVisualizer(system)
visualizer.visualize()

In [ ]:
system.initialize(t)
pendulum.setup_monitoring()
pendulum.display_monitoring()
pendulum._update_output_states(t=0)

In [ ]:
t_start = 0.0
t_end = pendulum.fem.sim_params.t_end
dt = 0.01  # Macro step size

system.run(t0=t_start, tf=t_end, dt=dt)

In [ ]:
# 7) Extract and plot results
history = system.get_history()
pendulum_history = history["MasterPendulum"]
event_history = history["Events"]

t_vals, data = pendulum_history
q_vals = data["theta"]
omega_vals = data["omega"]
alpha_vals = data["alpha"]

# Get event times
event_times = event_history.get(("MasterPendulum", "wall_hit"), [])
t_event = event_times[1] if event_times else None

print(f"\nDetected {len(event_times)} events")
if t_event:
    print(f"First event at t = {t_event.t:.8f} s")

In [ ]:
marker_size = 3
fem_style = {
    "color": "#063ccf",
    "linestyle": "None",
    "marker": "o",
    "markersize": marker_size,
    "linewidth": 3.5,
    "label": "FEM",
}
opensim_style = {
    "color": "#f8c20f",
    "linestyle": "None",
    "marker": "s",
    "markersize": marker_size,
    "linewidth": 3.5,
    "label": "OpenSim",
}  # Green
eqb_style = {
    "color": "#24aa09",
    "linestyle": "None",
    "marker": "^",
    "markersize": marker_size,
    "linewidth": 3.5,
    "label": "FMU",
}  # Purple
fontdict = {"fontsize": 11, "fontweight": "bold"}

t_vals = {mode: [] for mode in pendulum.models.keys()}
torque_vals = {mode: [] for mode in pendulum.models.keys()}
alpha_vals = {mode: [] for mode in pendulum.models.keys()}
omega_vals = {mode: [] for mode in pendulum.models.keys()}
q_vals = {mode: [] for mode in pendulum.models.keys()}

for mode in pendulum.models.keys():
    t_vals[mode], res_dict = pendulum.models[mode].get_history_arrays()
    alpha_vals[mode] = res_dict["alpha"]
    omega_vals[mode] = res_dict["omega"]
    q_vals[mode] = res_dict["theta"]

In [ ]:
# 8) Visualization
fig, axs = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle(
    "FEM Pendulum: Event Localization using Hybrid Algorithm",
    fontweight="bold",
    fontsize=14,
    y=1.02,
)

# Top left: Full angle trajectory
(line0, ) = axs[0, 0].plot(t_ref, q_ref, color="red", linestyle="-", linewidth=2, label="Reference")
(line1,) = axs[0, 0].plot(t_vals["FEM"], q_vals["FEM"], **fem_style)
(line2,) = axs[0, 0].plot(t_vals["OpenSim"], q_vals["OpenSim"], **opensim_style)
(line3,) = axs[0, 0].plot(t_vals["FMU"], q_vals["FMU"], **eqb_style)
line4 = axs[0, 0].axhline(0, color="k", linestyle="--", label="Wall Position")
if t_event:
    line3 = axs[0, 0].axvline(
        t_event.t, color="red", linestyle=":", linewidth=2, label="Event Time"
    )
axs[0, 0].set_xlabel("Time [s]")
axs[0, 0].set_ylabel("Angle [rad]")
axs[0, 0].grid()

# Top right: Zoomed angle near event
if t_event:
    zoom_window = 0.03
    mask = (t_vals["FEM"] >= t_event.t - zoom_window) & (t_vals["FEM"] <= t_event.t + zoom_window)
    mask_ref = (t_ref >= t_event.t - zoom_window) & (t_ref <= t_event.t + zoom_window)
    axs[0, 1].plot(t_ref[mask_ref], q_ref[mask_ref], color="red", linestyle="-", linewidth=4)
    axs[0, 1].plot(t_vals["FEM"][mask], q_vals["FEM"][mask], "o-", markersize=6, color="#063ccf")
    axs[0, 1].axhline(0, color="k", linestyle="--")
    axs[0, 1].axvline(t_event.t, color="red", linestyle=":", linewidth=2)
    axs[0, 1].set_xlabel("Time [s]")
    axs[0, 1].set_ylabel("Angle [rad]")
    axs[0, 1].set_title(f"Zoomed: Event at t={t_event.t:.4f} s")
    axs[0, 1].grid()

# Bottom left: Full angular velocity
(line8,) = axs[1, 0].plot(t_ref, omega_ref, color="red", linestyle="-", linewidth=2, label="Reference")
(line5,) = axs[1, 0].plot(t_vals["FEM"], omega_vals["FEM"], **fem_style)
(line6,) = axs[1, 0].plot(t_vals["OpenSim"], omega_vals["OpenSim"], **opensim_style)
(line7,) = axs[1, 0].plot(t_vals["FMU"], omega_vals["FMU"], **eqb_style)

if t_event:
    axs[1, 0].axvline(t_event.t, color="red", linestyle=":", linewidth=2)
axs[1, 0].set_xlabel("Time [s]")
axs[1, 0].set_ylabel("Angular Velocity [rad/s]")
axs[1, 0].grid()

# Bottom right: Zoomed velocity near event
if t_event:
    axs[1, 1].plot(t_vals["FEM"][mask], omega_vals["FEM"][mask], "o-", color="#063ccf", markersize=6)
    axs[1, 1].axvline(t_event.t, color="red", linestyle=":", linewidth=2)
    axs[1, 1].set_xlabel("Time [s]")
    axs[1, 1].set_ylabel("Angular Velocity [rad/s]")
    axs[1, 1].set_title("Zoomed: Velocity Inversion")
    axs[1, 1].grid()

signals = [line0, line1, line4, line2, line3]  # , line5]
labels = [s.get_label() for s in signals if s is not None]
fig.legend(signals, labels, loc="upper center", bbox_to_anchor=(0.5, 1), ncol=5)

plt.tight_layout()
#plt.savefig("../figures/Hybrid/test_2_location_FEM.svg")
plt.show()